# SAADE–AUA v9 | Незалежна перевірка AASIST / AASIST-L

**Мета:** встановити, чи ONNX-інференс у v9 має правильну орієнтацію REAL/FAKE та чи збігається з еталонною PyTorch-реалізацією **тих самих pretrained-моделей**. **Жодного навчання, жодного hidden test, жодного повторного прогону всього AUA.**

Перевіряємо на 12 парах із `MATCHED_TRAIN` (24 аудіофайли) для кожної з двох моделей. Порівнюємо **ідентичний** вхідний сигнал ONNX і PyTorch у двох варіантах передобробки: як у v9 (нульове доповнення до 64 600) та як у документації моделі (повторення до 64 600). Всі контрольні суми та результати записуються в окрему папку під `results`.

**Еталонні джерела:** [AASIST](https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST), [AASIST-L](https://huggingface.co/SpeechAntiSpoofingBenchmarks/AASIST-L). Еталонний `logits[:, 1]` — **bona fide** (REAL), тому score FAKE = `-logits[:, 1]`.

> Увага: для побудови еталонної PyTorch-моделі імпортується код `_net.py` із публічного репозиторію Hugging Face. Перегляньте вміст джерела, якщо не довіряєте його власнику. Не завантажуйте невідомі checkpoint; `torch.load` викликається з `weights_only=True`.


## 1. Налаштування та залежності

In [ ]:
from pathlib import Path
import os, sys, json, ast, math, hashlib, urllib.request, importlib.util, sqlite3
import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal
from fractions import Fraction
from sklearn.metrics import roc_auc_score

PROJECT = Path('/home/jovyan/work/audio_real_fake')
BASE = PROJECT/'results'/'SAADE_AUA_2026_v9_COMPLETE_EVALUATION'
RUN = BASE/'FULL_EXPERIMENT'
CACHE = BASE/'pretrained_model_cache'
OUT = BASE/'AASIST_REFERENCE_PARITY_AUDIT'
OUT.mkdir(parents=True, exist_ok=True)
V6_DIR = next((p for p in [PROJECT/'results'/'SAADE_AUA_2026_v6_2_MATCHED_TTS',
                         PROJECT/'results'/'SAADE_AUA_2026_v6_MATCHED_TTS']
               if (p/'V6_LOCKED_MATCHED_SPLITS.csv').exists()), None)
assert V6_DIR is not None, 'Не знайдено matched REAL–TTS маніфест'
assert CACHE.exists() and (RUN/'03_MODEL_PROVENANCE.json').exists(), 'Спочатку має існувати завершений v9'
N_PAIRS = 12
MODELS = {
    'AASIST_ONNX': {'repo':'SpeechAntiSpoofingBenchmarks/AASIST', 'onnx':'aasist.onnx',
                    'script':'aasist.py', 'weights':'AASIST.pth'},
    'AASIST_L_ONNX': {'repo':'SpeechAntiSpoofingBenchmarks/AASIST-L', 'onnx':'aasist-l.onnx',
                      'script':'aasist_l.py', 'weights':'AASIST-L.pth'},
}
print('Перевірка тільки MATCHED_TRAIN. Жоден hidden-файл не читається.')


## 2. Безпечне відновлення реальних аудіопар з train

In [ ]:
manifest = pd.read_csv(V6_DIR/'V6_LOCKED_MATCHED_SPLITS.csv', low_memory=False)
train = manifest.loc[manifest['split'].astype(str).eq('train')].copy()
train['pair_id'] = train['pair_id'].astype(str)
train['label'] = train['label'].astype(int)
assert set(train['label']) == {0,1}
assert train.groupby('pair_id')['label'].agg(['sum','count']).eq([1,2]).all().all(), 'Неправильна структура пар'
pair_ids = np.random.default_rng(20261117).choice(
    np.asarray(sorted(train.pair_id.unique())), size=min(N_PAIRS,train.pair_id.nunique()), replace=False)
selected = train.loc[train.pair_id.isin(pair_ids)].copy()
assert len(selected) == 2*len(pair_ids)
assert not selected['split'].astype(str).eq('hidden_test').any()

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(2*1024*1024), b''): h.update(b)
    return h.hexdigest()

def decode_exact_v9(path, start_sec, active_n):
    # Відтворюємо read_wave() із v9: завжди 4 с за 16 кГц, доповнення нулями до 64000.
    with sf.SoundFile(str(path)) as f:
        rate=f.samplerate
        f.seek(max(0,int(round(float(start_sec)*rate))))
        x=f.read(frames=int(math.ceil(4.0*rate)),dtype='float32',always_2d=True).mean(axis=1)
    if rate!=16000:
        ratio=Fraction(16000,rate)
        x=signal.resample_poly(x,ratio.numerator,ratio.denominator)
    x=np.nan_to_num(np.asarray(x,dtype=np.float32),nan=0,posinf=0,neginf=0)
    x=x[:int(active_n)]
    return np.pad(x,(0,max(0,64000-len(x))))[:64000].astype(np.float32)

samples=[]
for pid in pair_ids:
    duo=selected.loc[selected.pair_id.eq(pid)]
    starts=[]; available=[]
    for r in duo.itertuples(index=False):
        assert Path(r.path).is_file(),f'Аудіо відсутнє: {r.path}'
        with sf.SoundFile(str(r.path)) as f:
            available.append(float(len(f)/f.samplerate)-float(getattr(r,'segment_start_sec',0) or 0))
        starts.append(float(getattr(r,'segment_start_sec',0) or 0))
    active_n=min(64000,max(1,int(math.floor(min(available)*16000))))
    assert active_n>=8000,'Надто коротка пара для перевірки'
    for r in duo.itertuples(index=False):
        w=decode_exact_v9(r.path,float(getattr(r,'segment_start_sec',0) or 0),active_n)
        samples.append({'pair_id':pid,'label':int(r.label),
                        'sample_id':f'matched:{pid}:{int(r.label)}','wave':w,'active_n':active_n})
print('Діагностичних аудіозаписів:',len(samples), '| REAL:',sum(x['label']==0 for x in samples),
      '| FAKE:',sum(x['label']==1 for x in samples))


## 3. Завантаження еталонної мережі та checkpoint (не нової моделі)

In [ ]:
import torch
import onnxruntime as ort

def download_reference(repo,filename,destination,min_size=100):
    destination.parent.mkdir(parents=True,exist_ok=True)
    if destination.exists() and destination.stat().st_size>=min_size:
        return destination
    url=f'https://huggingface.co/{repo}/resolve/main/{filename}'
    req=urllib.request.Request(url,headers={'User-Agent':'SAADE-parity-audit/1.0','Accept-Encoding':'identity'})
    tmp=destination.with_suffix(destination.suffix+'.partial')
    try:
        with urllib.request.urlopen(req,timeout=180) as resp, open(tmp,'wb') as f:
            while True:
                chunk=resp.read(1<<20)
                if not chunk:break
                f.write(chunk)
        assert tmp.stat().st_size>=min_size, f'Неповний файл: {url}'
        os.replace(tmp,destination)
    finally:tmp.unlink(missing_ok=True)
    return destination

def parse_reference_args(script):
    tree=ast.parse(Path(script).read_text(encoding='utf8'))
    for node in tree.body:
        if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='_D_ARGS' for t in node.targets):
            return ast.literal_eval(node.value)
    raise RuntimeError('Не знайдено _D_ARGS у еталонній реалізації: '+str(script))

def load_pytorch_reference(model_name,cfg):
    root=OUT/'reference_sources'/model_name
    src=download_reference(cfg['repo'],'_net.py',root/'_net.py',10000)
    script=download_reference(cfg['repo'],cfg['script'],root/cfg['script'],500)
    weight=download_reference(cfg['repo'],cfg['weights'],root/cfg['weights'],100000)
    args=parse_reference_args(script)
    module_name=f'_saade_audit_net_{model_name}'
    spec=importlib.util.spec_from_file_location(module_name,src)
    mod=importlib.util.module_from_spec(spec)
    sys.modules[module_name]=mod
    spec.loader.exec_module(mod)
    net=mod.Model(args)
    state=torch.load(weight,map_location='cpu',weights_only=True)
    if isinstance(state,dict) and 'state_dict' in state:state=state['state_dict']
    net.load_state_dict(state,strict=True)
    return net.eval().cpu(), {'net_py_sha256':sha256(src),'wrapper_sha256':sha256(script),'pth_sha256':sha256(weight)}

existing_provenance=json.loads((RUN/'03_MODEL_PROVENANCE.json').read_text(encoding='utf8'))
print('Зчитано SHA-256 checkpoint із v9:',list(existing_provenance))


## 4. Ідентичний вхід → ONNX проти PyTorch; перевірка нульового та повторного padding

In [ ]:
def to_64600(w,active_n,method):
    if method=='v9_zeros':
        return np.pad(w[:64000],(0,600)).astype(np.float32)
    if method=='reference_repeat':
        x=w[:active_n]
        assert len(x)>0
        return np.tile(x,int(np.ceil(64600/len(x))))[:64600].astype(np.float32)
    raise ValueError(method)

def onnx_logits(session,x):
    meta=session.get_inputs()[0]
    if len(meta.shape)==2: inp=x[None,:]
    elif len(meta.shape)==3 and meta.shape[1] in (1,None,'1'): inp=x[None,None,:]
    else:raise RuntimeError('Непідтримуваний ONNX shape: '+str(meta.shape))
    out=session.run(None,{meta.name:inp})
    logits=np.asarray(out[0],dtype=np.float64).reshape(-1)
    assert logits.size==2,f'Очікували 2 логіти, отримали {logits.size}'
    return logits

rows=[]; summary=[]
for name,cfg in MODELS.items():
    onnx_fp=CACHE/cfg['onnx']
    assert onnx_fp.is_file(),f'Не знайдено ONNX з v9: {onnx_fp}'
    actual_sha=sha256(onnx_fp)
    expected_sha=existing_provenance[name]['weights_sha256']
    assert actual_sha==expected_sha, f'ONNX модель змінилася з часу v9: {name}'
    net,meta=load_pytorch_reference(name,cfg)
    session=ort.InferenceSession(str(onnx_fp), providers=['CPUExecutionProvider'])
    for s in samples:
        for padding in ('v9_zeros','reference_repeat'):
            x=to_64600(s['wave'],s['active_n'],padding)
            y_onnx=onnx_logits(session,x)
            with torch.inference_mode():
                _,y_pt=net(torch.from_numpy(x)[None,:])
            y_pt=y_pt.detach().cpu().numpy().reshape(-1).astype(float)
            assert y_pt.size==2 and np.isfinite(y_pt).all()
            rows.append(dict(model=name,pair_id=s['pair_id'],label=s['label'],sample_id=s['sample_id'],
                padding=padding,onnx_bonafide=float(y_onnx[1]),pytorch_bonafide=float(y_pt[1]),
                onnx_fake_score=float(-y_onnx[1]),pytorch_fake_score=float(-y_pt[1]),
                abs_diff=float(abs(y_onnx[1]-y_pt[1])),wrong_index_diff=float(abs(y_onnx[0]-y_pt[1])),
                onnx_class0=float(y_onnx[0]),torch_class0=float(y_pt[0]),active_n=s['active_n']))
    del net,session
    print(name,'перевірено. PyTorch checkpoint SHA:',meta['pth_sha256'])

result=pd.DataFrame(rows)
result.to_csv(OUT/'01_ONNX_PYTORCH_PARITY_PER_RECORD.csv',index=False)
from scipy.stats import spearmanr
for (model,padding),d in result.groupby(['model','padding']):
    a=d['onnx_bonafide'].to_numpy(); b=d['pytorch_bonafide'].to_numpy()
    rho=float(spearmanr(a,b).statistic) if len(np.unique(a))>1 and len(np.unique(b))>1 else float('nan')
    delta=float(np.max(np.abs(a-b)))
    close=bool(np.allclose(a,b,rtol=1e-2,atol=2e-2))
    orientation=(d['wrong_index_diff'].median() > d['abs_diff'].median())
    summary.append(dict(model=model,padding=padding,N=len(d),max_abs_diff=delta,spearman=rho,
                        outputs_numerically_close=close,index1_closer_than_index0=bool(orientation),
                        status='PASS' if close and (np.isnan(rho) or rho>0.999) and orientation else 'CHECK'))
report=pd.DataFrame(summary)
report.to_csv(OUT/'02_PARITY_SUMMARY.csv',index=False)
display(report)
assert (report.status=='PASS').all(), 'Є розбіжність PyTorch/ONNX: дивіться CSV — НЕ запускайте hidden test'
print('PASS: ONNX відтворює еталонну PyTorch-модель для однакових входів.')


## 5. Вплив типу padding і наявність старих прогнозів у v9

In [ ]:
pad=result.pivot_table(index=['model','sample_id','label'],columns='padding',values='onnx_fake_score').reset_index()
pad['abs_padding_effect']=(pad['v9_zeros']-pad['reference_repeat']).abs()
impact=pad.groupby('model')['abs_padding_effect'].agg(['count','median','max']).reset_index()
impact.columns=['model','n','median_abs_change','max_abs_change']
impact.to_csv(OUT/'03_PADDING_EFFECT.csv',index=False)
display(impact)

db_path=RUN/'05_FULL_PREDICTIONS.sqlite3'
if db_path.exists():
    con=sqlite3.connect(f'file:{db_path}?mode=ro',uri=True)
    checks=[]
    for s in samples:
        for name in MODELS:
            old=con.execute('SELECT score_fake FROM predictions WHERE model=? AND cohort=? AND sample_id=? AND scenario=?',
                (name,'MATCHED_TRAIN',s['sample_id'],'clean')).fetchone()
            if old:
                new=pad.loc[(pad.model==name)&(pad.sample_id==s['sample_id']),'v9_zeros'].iloc[0]
                checks.append({'model':name,'sample_id':s['sample_id'],'existing_score_fake':old[0],
                               'recomputed_score_fake':float(new),'abs_diff':abs(old[0]-new)})
    con.close()
    oldcheck=pd.DataFrame(checks)
    oldcheck.to_csv(OUT/'04_EXISTING_V9_SCORE_CHECK.csv',index=False)
    display(oldcheck.groupby('model').abs_diff.agg(['count','max']) if len(oldcheck) else 'Не знайдено MATCHED_TRAIN clean у SQLite')
    if len(oldcheck):
        assert oldcheck.abs_diff.max()<0.02, 'Попередні v9 прогнози не збігаються з поточним ONNX на тих самих входах'
else:print('SQLite недоступний: перевірка наявних score пропущена; parity ONNX/PyTorch усе одно виконана.')

print('Готово. Артефакти лише тут:',OUT)
print('Висновок:')
print('Якщо parity PASS та padding_effect мізерний — перевірка ONNX вважається успішною.')
print('Якщо padding_effect великий — перед FREEZE потрібен окремий план перерахунку або обмеження висновків.')
